# Translate Text Files to English

This notebook translates multiple text files from a folder to English using deep-translator and saves them to an `/en` subfolder.
https://pypi.org/project/deep-translator/

In [2]:
import os
from pathlib import Path
from deep_translator import GoogleTranslator
import time

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
def translate_text_to_english(text, source_lang='fr', max_chunk_size=4500):
    """
    Translate text to English using Google Translator.
    Handles long texts by splitting into chunks.
    
    Args:
        text (str): Text to translate
        source_lang (str): Source language code (default: 'fr' for French)
        max_chunk_size (int): Maximum characters per chunk
        
    Returns:
        str: Translated text
    """
    if not text or len(text.strip()) == 0:
        return text
    
    translator = GoogleTranslator(source=source_lang, target='en')
    
    # If text is short, translate directly
    if len(text) <= max_chunk_size:
        try:
            return translator.translate(text)
        except Exception as e:
            print(f"Translation error: {e}")
            return text
    
    # Split long text into chunks by paragraphs
    paragraphs = text.split('\n\n')
    translated_paragraphs = []
    current_chunk = ""
    
    for para in paragraphs:
        if len(current_chunk) + len(para) < max_chunk_size:
            current_chunk += para + "\n\n"
        else:
            if current_chunk:
                try:
                    translated_paragraphs.append(translator.translate(current_chunk.strip()))
                    time.sleep(0.5)  # Avoid rate limiting
                except Exception as e:
                    print(f"Translation error: {e}")
                    translated_paragraphs.append(current_chunk)
            current_chunk = para + "\n\n"
    
    # Translate remaining chunk
    if current_chunk:
        try:
            translated_paragraphs.append(translator.translate(current_chunk.strip()))
        except Exception as e:
            print(f"Translation error: {e}")
            translated_paragraphs.append(current_chunk)
    
    return "\n\n".join(translated_paragraphs)

print("Translation function defined!")

Translation function defined!


## Set Input Folder Path

Specify the folder containing the French text files to translate.

In [4]:
# Set your input folder path here
input_folder = r"../data/raw/macommunaute/fr"

# Verify the folder exists
input_path = Path(input_folder)
if input_path.exists():
    print(f"✓ Input folder found: {input_path.absolute()}")
    txt_files = list(input_path.glob("*.txt"))
    print(f"✓ Found {len(txt_files)} .txt files")
    for f in txt_files:
        print(f"  - {f.name}")
else:
    print(f"✗ Folder not found: {input_path.absolute()}")

✓ Input folder found: d:\mscudemproject\main\dynamic_rags\notebooks\..\data\raw\macommunaute\fr
✓ Found 11 .txt files
  - sports_loisirs.txt
  - sante.txt
  - retraite.txt
  - logement.txt
  - justice_droits.txt
  - emploi.txt
  - education.txt
  - alimentation.txt
  - aide_ecoute.txt
  - communautes_culturelles_nouveaux_arrivants.txt
  - famille.txt


## Translate TXT Files

Translate all .txt files in the input folder to English and save them to the `/en` subfolder.

In [5]:
# Create output folder
output_folder = input_path / "../en"
output_folder.mkdir(exist_ok=True)
print(f"✓ Output folder ready: {output_folder.absolute()}\n")

✓ Output folder ready: d:\mscudemproject\main\dynamic_rags\notebooks\..\data\raw\macommunaute\fr\..\en



In [23]:
# Process TXT files
input_txt_files = list(input_path.glob("*.txt"))
print(f"Found {len(input_txt_files)} .txt files to translate.")

total_files = len(input_txt_files)
for idx, txt_file in enumerate(input_txt_files, 1):
    print(f"[{idx}/{total_files}] Processing TXT: {txt_file.name}")

    try:
        with open(txt_file, 'r', encoding='utf-8') as f:
            original_text = f.read()

        print(f"  - Original size: {len(original_text)} characters")

        translated_text = translate_text_to_english(original_text, source_lang='fr')

        print(f"  - Translated size: {len(translated_text)} characters")

        output_file = output_folder / txt_file.name
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(translated_text)

        print(f"  ✓ Saved to: {output_file.name}\n")
    except Exception as e:
        print(f"  ✗ Error processing {txt_file.name}: {e}\n")

print("=" * 60)
print(f"✓ TXT translation complete! {total_files} files processed.")
print(f"✓ Results saved to: {output_folder.absolute()}")

Found 11 .txt files to translate.
[1/11] Processing TXT: sports_loisirs.txt
  - Original size: 64985 characters
  - Translated size: 59525 characters
  ✓ Saved to: sports_loisirs.txt

[2/11] Processing TXT: sante.txt
  - Original size: 134202 characters
  - Translated size: 120151 characters
  ✓ Saved to: sante.txt

[3/11] Processing TXT: retraite.txt
  - Original size: 2303 characters
  - Translated size: 2114 characters
  ✓ Saved to: retraite.txt

[4/11] Processing TXT: logement.txt
  - Original size: 27578 characters
  - Translated size: 25150 characters
  ✓ Saved to: logement.txt

[5/11] Processing TXT: justice_droits.txt
  - Original size: 12967 characters
  - Translated size: 11917 characters
  ✓ Saved to: justice_droits.txt

[6/11] Processing TXT: emploi.txt
  - Original size: 15401 characters
  - Translated size: 13715 characters
  ✓ Saved to: emploi.txt

[7/11] Processing TXT: education.txt
  - Original size: 62766 characters
  - Translated size: 57291 characters
  ✓ Saved to:

## Translate PDF Files

Translate text-based French PDFs from the input folder into English PDFs saved to `/en`.

In [6]:
#!pip install pdfplumber reportlab

In [8]:
import pdfplumber
from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from xml.sax.saxutils import escape

input_pdf_files = list(input_path.glob("*.pdf"))
print(f"Found {len(input_pdf_files)} PDF files to translate.\n")

styles = getSampleStyleSheet()

for idx, pdf_file in enumerate(input_pdf_files, 1):
    print(f"[{idx}/{len(input_pdf_files)}] Processing PDF: {pdf_file.name}")

    try:
        output_pdf = output_folder / f"{pdf_file.stem}_en.pdf"
        doc = SimpleDocTemplate(str(output_pdf), pagesize=A4)

        story = []

        with pdfplumber.open(pdf_file) as pdf:
            total_pages = len(pdf.pages)
            print(f"  - Pages detected: {total_pages}")

            for page_idx, page in enumerate(pdf.pages, 1):
                page_text = page.extract_text()

                if not page_text or not page_text.strip():
                    print(f"    Page {page_idx}: empty or scanned")
                    story.append(Paragraph("[No extractable text]", styles["Normal"]))
                else:
                    # Translate THIS PAGE ONLY
                    translated_page = translate_text_to_english(
                        page_text,
                        source_lang="fr"
                    )

                    # Write translated page
                    for paragraph in translated_page.split("\n\n"):
                        safe_text = escape(paragraph)
                        story.append(Paragraph(safe_text, styles["Normal"]))

                # Force page break (except after last page)
                if page_idx < total_pages:
                    story.append(PageBreak())

        doc.build(story)
        print(f"  ✓ Saved PDF: {output_pdf.name}\n")

    except Exception as e:
        print(f"  ✗ Error processing {pdf_file.name}: {e}\n")

print("✓ PDF → PDF translation complete (page-preserving)!")

Found 11 PDF files to translate.

[1/11] Processing PDF: sports_loisirs.pdf
  - Pages detected: 37
  ✓ Saved PDF: sports_loisirs_en.pdf

[2/11] Processing PDF: sante.pdf
  - Pages detected: 65
  ✓ Saved PDF: sante_en.pdf

[3/11] Processing PDF: retraite.pdf
  - Pages detected: 1
  ✓ Saved PDF: retraite_en.pdf

[4/11] Processing PDF: logement.pdf
  - Pages detected: 17
  ✓ Saved PDF: logement_en.pdf

[5/11] Processing PDF: justice_droits.pdf
  - Pages detected: 7
  ✓ Saved PDF: justice_droits_en.pdf

[6/11] Processing PDF: emploi.pdf
  - Pages detected: 8
  ✓ Saved PDF: emploi_en.pdf

[7/11] Processing PDF: education.pdf
  - Pages detected: 34
  ✓ Saved PDF: education_en.pdf

[8/11] Processing PDF: alimentation.pdf
  - Pages detected: 7
  ✓ Saved PDF: alimentation_en.pdf

[9/11] Processing PDF: aide_ecoute.pdf
  - Pages detected: 32
  ✓ Saved PDF: aide_ecoute_en.pdf

[10/11] Processing PDF: communautes_culturelles_nouveaux_arrivants.pdf
  - Pages detected: 40
  ✓ Saved PDF: communautes_

In [25]:
# Translate French PDFs to English PDFs in the same output folder
# Requires: pip install pdfplumber reportlab

import textwrap

try:
    import pdfplumber
    from reportlab.pdfgen import canvas
    from reportlab.lib.pagesizes import LETTER
except ImportError:
    raise ImportError("Missing packages. Run: pip install pdfplumber reportlab")

input_pdf_files = list(input_path.glob("*.pdf"))
print(f"Found {len(input_pdf_files)} .pdf files to translate.")

font_size = 10
line_height = font_size + 2
max_chars_per_line = 110
margin = 50

for idx, pdf_file in enumerate(input_pdf_files, 1):
    print(f"[{idx}/{len(input_pdf_files)}] Processing PDF: {pdf_file.name}")

    output_pdf = output_folder / pdf_file.name

    with pdfplumber.open(pdf_file) as pdf:
        c = canvas.Canvas(str(output_pdf), pagesize=LETTER)
        width, height = LETTER

        for page_index, page in enumerate(pdf.pages, 1):
            text = page.extract_text() or ""
            translated_text = translate_text_to_english(text, source_lang="fr")

            c.setFont("Helvetica", font_size)
            y = height - margin

            lines = []
            for paragraph in translated_text.split("\n"):
                if paragraph.strip() == "":
                    lines.append("")
                    continue
                lines.extend(textwrap.wrap(paragraph, width=max_chars_per_line))

            for line in lines:
                if y < margin:
                    c.showPage()
                    c.setFont("Helvetica", font_size)
                    y = height - margin
                c.drawString(margin, y, line)
                y -= line_height

            if page_index < len(pdf.pages):
                c.showPage()

        c.save()

    print(f"  ✓ Saved translated PDF: {output_pdf.name}")

print("=" * 60)
print(f"✓ PDF translation complete! Results saved to: {output_folder.absolute()}")

Found 11 .pdf files to translate.
[1/11] Processing PDF: sports_loisirs.pdf
  ✓ Saved translated PDF: sports_loisirs.pdf
[2/11] Processing PDF: sante.pdf
  ✓ Saved translated PDF: sante.pdf
[3/11] Processing PDF: retraite.pdf
  ✓ Saved translated PDF: retraite.pdf
[4/11] Processing PDF: logement.pdf
  ✓ Saved translated PDF: logement.pdf
[5/11] Processing PDF: justice_droits.pdf
  ✓ Saved translated PDF: justice_droits.pdf
[6/11] Processing PDF: emploi.pdf
  ✓ Saved translated PDF: emploi.pdf
[7/11] Processing PDF: education.pdf
  ✓ Saved translated PDF: education.pdf
[8/11] Processing PDF: alimentation.pdf
  ✓ Saved translated PDF: alimentation.pdf
[9/11] Processing PDF: aide_ecoute.pdf
  ✓ Saved translated PDF: aide_ecoute.pdf
[10/11] Processing PDF: communautes_culturelles_nouveaux_arrivants.pdf
  ✓ Saved translated PDF: communautes_culturelles_nouveaux_arrivants.pdf
[11/11] Processing PDF: famille.pdf
  ✓ Saved translated PDF: famille.pdf
✓ PDF translation complete! Results saved t

## Verify Results (Optional)

Preview a sample of the translated content.

In [ ]:
# Read and display a sample from the first translated file
if txt_files:
    sample_file = output_folder / txt_files[0].name
    if sample_file.exists():
        with open(sample_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        print(f"Sample from: {sample_file.name}")
        print("=" * 60)
        print(content[:500])  # Show first 500 characters
        if len(content) > 500:
            print(f"\n... ({len(content) - 500} more characters)")
    else:
        print("No translated files found.")

Sample from: sports_loisirs.txt
===== Organization 1: The YMCAs of Quebec =====
URL: https://macommunaute.ca/bottin-des-agences/ymcaduquebec/
Website: http://www.ymcaquebec.org/

Description: ABOUT THE ORGANIZATION
Contact details
1435 Rue Drummond,
Montreal QC H3G 1W3
514-849-5331
514-849-5863
Opening hours
SCHEDULE
Hours vary from one YMCA center to another. Please contact the YMCA nearest to you.
Social networks
Branches (8)
THE YMCAs OF QUEBEC
1435 Rue Drummond
514-849-5331
514-849-5863
DOWNTOWN YMCA
1440 Stanley Street
51

... (59025 more characters)


: 